# 01 — Exploratory data analysis: Women's Clothing E-Commerce Reviews

Look at the data **before** writing ingestion code. What we find here decides:
- how ingestion must handle names and types (`02_ingestion`),
- which text goes into the vector database and which metadata goes with it (`03_vector_index`),
- which questions the chatbot can answer, and which need a clarifying question.

## Section 0 — Setup

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd().parent
RAW_FILE = PROJECT_ROOT / "data" / "raw" / "Womens Clothing E-Commerce Reviews.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_FILE exists:", RAW_FILE.exists())

PROJECT_ROOT: d:\ChatBot test
RAW_FILE exists: True


## Section 1 — Load and overview
The CSV is about 8 MB, so loading takes a few seconds. Exploration uses pandas' normal type guessing; ingestion (`02`) will read raw cells instead.

In [3]:
reviews_raw = pd.read_csv(RAW_FILE)

In [4]:
reviews_raw.shape

(23486, 11)

In [5]:
reviews_raw.head()

,Unnamed: 0,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


In [6]:
reviews_raw.dtypes

Unnamed: 0                 int64
Clothing ID                int64
Age                        int64
Title                        str
Review Text                  str
Rating                     int64
Recommended IND            int64
Positive Feedback Count    int64
Division Name                str
Department Name              str
Class Name                   str
dtype: object

In [7]:
reviews_raw.isna().sum()

Unnamed: 0                    0
Clothing ID                   0
Age                           0
Title                      3810
Review Text                 845
Rating                        0
Recommended IND               0
Positive Feedback Count       0
Division Name                14
Department Name              14
Class Name                   14
dtype: int64

### Q1 — What is the `Unnamed: 0` column?
Its header cell is empty, so pandas invents the name `Unnamed: 0`. If it's simply 0, 1, 2, … it is a **leftover pandas index** from whoever saved the file — but it is still useful as a stable, unique **review ID**.

In [8]:
review_index = reviews_raw["Unnamed: 0"]
print("Equals 0..n-1:", (review_index.values == range(len(reviews_raw))).all())
print("Unique:", review_index.is_unique)

Equals 0..n-1: True
Unique: True


### Q2 — Ratings and recommendations
`Recommended IND` is 1 when the reviewer recommends the product. Does it agree with the star rating?

In [9]:
reviews_raw["Rating"].value_counts().sort_index()

Rating
1      842
2     1565
3     2871
4     5077
5    13131
Name: count, dtype: int64

In [10]:
pd.crosstab(reviews_raw["Rating"], reviews_raw["Recommended IND"])

Recommended IND,0,1
Rating,,
1,826,16
2,1471,94
3,1682,1189
4,168,4909
5,25,13106


### Q3 — Categories: division, department, class

In [11]:
for col in ["Division Name", "Department Name", "Class Name"]:
    print(reviews_raw[col].value_counts(dropna=False), "\n")

Division Name
General           13850
General Petite     8120
Initmates          1502
NaN                  14
Name: count, dtype: int64 

Department Name
Tops        10468
Dresses      6319
Bottoms      3799
Intimate     1735
Jackets      1032
Trend         119
NaN            14
Name: count, dtype: int64 

Class Name
Dresses           6319
Knits             4843
Blouses           3097
Sweaters          1428
Pants             1388
Jeans             1147
Fine gauge        1100
Skirts             945
Jackets            704
Lounge             691
Swim               350
Outerwear          328
Shorts             317
Sleep              228
Legwear            165
Intimates          154
Layering           146
Trend              119
NaN                 14
Casual bottoms       2
Chemises             1
Name: count, dtype: int64 



In [12]:
pd.crosstab(reviews_raw["Department Name"], reviews_raw["Division Name"])

Division Name,General,General Petite,Initmates
Department Name,,,
Bottoms,2542,1257,0
Dresses,3730,2589,0
Intimate,0,233,1502
Jackets,645,387,0
Tops,6837,3631,0
Trend,96,23,0


In [13]:
# The 14 rows with no categories
reviews_raw[reviews_raw["Division Name"].isna()][["Clothing ID", "Rating", "Title", "Review Text"]]

,Clothing ID,Rating,Title,Review Text
9444,72,5,My favorite socks!!!,"I never write reviews, but these socks are so ..."
13767,492,5,So soft!,I just love this hoodie! it is so soft and com...
13768,492,5,Wardrobe staple,Love this hoodie. so soft and goes with everyt...
13787,492,5,NaN,NaN
16216,152,5,Warm and cozy,"Just what i was looking for. soft, cozy and warm."
16221,152,5,Love!,I am loving these. they are quite long but are...
16223,152,5,"""long and warm""",These leg warmers are perfect for me. they are...
18626,184,5,Nubby footless tights,"These are amazing quality. i agree, size up to..."
18671,184,5,New workhorse,These tights are amazing! if i care for them w...
20088,772,5,Comfy sweatshirt!,This sweatshirt is really nice! it's oversize...


### Q4 — Are the categories a property of the product?
If every `Clothing ID` always has the same department and class, they describe the **product**. If not, they describe the **review** (for example, which size range was bought).

In [14]:
categories = ["Division Name", "Department Name", "Class Name"]
per_product = reviews_raw.dropna(subset=categories).groupby("Clothing ID")[categories].nunique()
print("Products with more than one value:")
print((per_product > 1).sum())

Products with more than one value:
Division Name      196
Department Name      0
Class Name           1
dtype: int64


In [15]:
reviews_raw["Clothing ID"].value_counts().describe()

count    1206.000000
mean       19.474295
std        69.009764
min         1.000000
25%         1.000000
50%         2.000000
75%         6.750000
max      1024.000000
Name: count, dtype: float64

### Q5 — Text columns: what will go into the vector database?
`Title` and `Review Text` are the free text. We need to know how often each is blank, how long the reviews are, and whether there are duplicates.

In [16]:
title_blank = reviews_raw["Title"].isna()
text_blank = reviews_raw["Review Text"].isna()
pd.Series({
    "title and text": (~title_blank & ~text_blank).sum(),
    "text only": (title_blank & ~text_blank).sum(),
    "title only": (~title_blank & text_blank).sum(),
    "neither": (title_blank & text_blank).sum(),
})

title and text    19675
text only          2966
title only            1
neither             844
dtype: int64

In [17]:
word_counts = reviews_raw["Review Text"].dropna().str.split().str.len()
word_counts.describe()

count    22641.000000
mean        60.196679
std         28.534612
min          2.000000
25%         36.000000
50%         59.000000
75%         88.000000
max        115.000000
Name: Review Text, dtype: float64

In [18]:
print("Duplicate review texts:", reviews_raw["Review Text"].dropna().duplicated().sum())
print("Duplicate rows (ignoring the index column):", reviews_raw.drop(columns="Unnamed: 0").duplicated().sum())

Duplicate review texts: 7
Duplicate rows (ignoring the index column): 21


### Q6 — Numeric columns and a first SQL-style question

In [19]:
reviews_raw[["Age", "Positive Feedback Count"]].describe()

,Age,Positive Feedback Count
count,23486.000000,23486.000000
mean,43.198544,2.535936
std,12.279544,5.702202
min,18.000000,0.000000
25%,34.000000,0.000000
50%,41.000000,1.000000
75%,52.000000,3.000000
max,99.000000,122.000000


In [20]:
# Preview of a "SQL first, then text" question: which department has the most negative reviews?
negative = reviews_raw[reviews_raw["Rating"] <= 2]
summary = reviews_raw.groupby("Department Name").agg(
    reviews=("Rating", "size"),
    avg_rating=("Rating", "mean"),
)
summary["negative_reviews"] = negative.groupby("Department Name").size()
summary.sort_values("negative_reviews", ascending=False).round(2)

,reviews,avg_rating,negative_reviews
Department Name,,,
Tops,10468,4.17,1114
Dresses,6319,4.15,689
Bottoms,3799,4.29,320
Intimate,1735,4.28,153
Jackets,1032,4.26,109
Trend,119,3.82,22


## Findings

**Dataset:** 23,486 reviews × 11 columns, one CSV (one "sheet"). 1,206 products (`Clothing ID`). **No date column.**

1. **`Unnamed: 0` is a leftover pandas index** (exactly 0…23,485, unique). Ingestion will see a **blank header**; we keep the column and rename it **`review_id`**. It is the key that links SQLite rows to vector-database results.

2. **Ratings are skewed positive:** 5★ 13,131 · 4★ 5,077 · 3★ 2,871 · 2★ 1,565 · 1★ 842.
   - `Recommended IND` mostly agrees with the rating, with exceptions (16 one-star reviews recommend the product; 25 five-star reviews don't).
   - "Negative review" is not defined in the data. **Rating ≤ 2** gives 2,407 reviews; the chatbot must state this assumption (or ask).

3. **Categories:** 3 divisions, 6 departments, 20 classes; **14 rows have no categories** (socks, hoodies, leg warmers…).
   - The division is spelled **`Initmates`** (typo in the source). Keep it as-is, but questions about "intimates" must match it — a good reason to show the LLM real sample values from the catalog.
   - `Department Name` = `Intimate`, `Class Name` = `Intimates`: three similar spellings for related ideas.

4. **Department and class describe the product** (0 and 1 inconsistent products). **Division does not:** 196 products appear under both `General` and `General Petite`, so division describes the review (size range bought).
   - Reviews per product: median 2, maximum 1,024.

5. **Text:** 19,675 reviews have title and text, 2,966 text only, 1 title only, **844 neither**.
   - Reviews are short: 2–115 words (median 59). Title + text fits in all-MiniLM's input limit without splitting into chunks: **one review = one vector**.
   - 7 duplicate review texts and 21 duplicate rows; reported, not removed.

6. **Numbers:** `Age` 18–99 (median 41); `Positive Feedback Count` 0–122 (median 1).
   - **Tops** has the most negative reviews (1,114) because it has the most reviews; **Trend** has the lowest average rating (3.82). "Most negative" can mean count or rate — another assumption to state.

### What this means for the next notebooks
- **Ingestion (`02`):** blank first header → rename to `review_id`; `Recommended IND` (0/1) stays an integer; no date handling needed.
- **Vector index (`03`):** document = `Title` + `Review Text`; skip the 844 empty rows; store `review_id`, `rating`, `recommended`, `department`, `class`, `division`, `age`, `clothing_id` as metadata so searches can be filtered.
- **Router:** time-based questions ("last month") are **unsupported** — there are no dates.